In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import fcluster
import matplotlib.pyplot as plt

from src.global_settings import settings
from src.sae import SparseAutoencoder
from src.activations import ActivationDataset
from src.utils.model_loader import load_sae
from src.features import load_score_data
from src.utils.explanation_loader import load_feature_explanation
from src.utils.metrics import (
    calculate_cdf,
    calculate_significances,
    calculate_pcc,
)
from src.utils.wordclouds import collect_cluster_tokens, show_all_wordclouds
from src.utils.clustering import visualize_clusters, cluster_with_protection, get_activation_distance_matrix, show_heatmap
from src.utils.font import setup_plt_font
setup_plt_font()

In [ ]:
META_DIR = Path(settings.feature_meta_dir)
RECORD_DIR = Path(settings.feature_record_dir)
EXPLANATION_DIR = Path(settings.feature_explanation_dir)

MODEL = "f3072_k8"
CHECKPOINT = f"{MODEL}_alpha"

LAYER = 21

meta_df = pd.read_csv(settings.text_dataset_path)

scores_data = load_score_data(checkpoint=CHECKPOINT, layer=LAYER, metric_name="total_post")
scores_df = pd.DataFrame(scores_data)

sample_counts = (scores_df > 0).sum(axis=0)
sample_counts = sample_counts[sample_counts > 0]

MIN_ACTIVE_SAMPLES = 200
live_feature_indices = sample_counts[sample_counts > MIN_ACTIVE_SAMPLES].index.tolist()

In [ ]:
probs = []
for _, row in meta_df.iterrows():
    label = row["label"]
    logit = row["logit"]
    
    if label == 0:
        prob = 1 - logit
    else:
        prob = logit
    
    probs.append(prob)

meta_df["prob"] = probs

meta_df.head()

In [ ]:
# meta_df.to_csv(str(settings.text_dataset_path), index=False)

In [ ]:
probs = meta_df["prob"].tolist()
preds = meta_df["pred"].tolist()
labels = meta_df["label"].tolist()

pl_signs = []
for p, l in zip(preds, labels):
    if p == l:
        pl_signs.append(1)
    else:
        pl_signs.append(-1)

signed_probs = [sgn * logit for sgn, logit in zip(pl_signs, probs)]

plt.hist(signed_probs, bins=50)
plt.show()

In [ ]:
# explanations = []
# descriptions = []
# discriminations = []
# determinations = []

# for feature_idx in tqdm(live_feature_indices):
#     explanation, description = load_feature_explanation(checkpoint=CHECKPOINT, layer=LAYER, feature_index=feature_idx)
#     explanations.append(explanation)
#     descriptions.append(description)
#     # discrimination, _, _ = calculate_discrimination(scores_df, meta_df, feature_idx)
#     # determination, _, _ = calculate_determination(scores_df, meta_df, feature_idx)
#     # discriminations.append(discrimination)
#     # determinations.append(determination)

# significances = calculate_significances(scores_df=scores_df, meta_df=meta_df)

# feature_df = pd.DataFrame({
#     "feature_index": live_feature_indices,
#     "explanation": explanations,
#     "description": descriptions,
#     "significance": significances[live_feature_indices].values,
#     # "discrimination": discriminations,
#     # "determination": determinations,
# })

# feature_df.to_csv("./outputs/analysis/live_features_f3072_k8_L21.csv", index=False)

feature_df = pd.read_csv("./outputs/analysis/live_features_f3072_k8_L21.csv")

In [ ]:
feature_df

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(range(len(feature_df["significance"])), feature_df["significance"], width=1, alpha=0.7, edgecolor='black')
plt.xlabel("Feature Index")
plt.ylabel("Significance Value")
plt.title(f"Significance Value for {CHECKPOINT} Layer {LAYER}")
plt.show()

### 1. 좋은 특징

In [ ]:
positive_features = feature_df[feature_df["significance"] > 0].sort_values(by="significance", ascending=False)
print(positive_features.to_markdown())

### 2. 나쁜 특징

In [ ]:
th = 0.1

sides = []
for feat in feature_df["feature_index"]:
    pcc = calculate_pcc(scores_df, meta_df, feat, key="pred")
    side = ""
    if pcc < -th:
        side = "👤"
    elif pcc > th:
        side = "🤖"
    else:
        side = "-"
    
    sides.append(side)

In [ ]:
top_k = 10

negative_features = feature_df[feature_df["significance"] < 0].sort_values(by="significance").head(top_k)
negative_features["side"] = sides
negative_features = negative_features.reset_index(drop=True)

cols = ["feature_index", "side", "explanation", "description", "significance"]

negative_features = negative_features[cols]
print(negative_features.to_markdown())

### 3. 흔한 특징

In [ ]:
topk = 15

freqs = []
for feature_idx in live_feature_indices:
    freq = (scores_df[feature_idx] > 0).sum() / len(scores_df)
    freq = round(freq, 2)
    freqs.append(freq)

feature_df["frequency"] = freqs
feature_df["side"] = sides

most_common_features = feature_df.sort_values(by="frequency", ascending=False).head(topk)

most_common_features = most_common_features.reset_index(drop=True)

cols = ["feature_index", "side", "frequency", "explanation", "description", "significance", "frequency"]
most_common_features = most_common_features[cols]

print(most_common_features.to_markdown())

# 거리 계산

In [ ]:
w1 = 0.5
w2 = 0.5

NUM_SAMPLES = 1000
SEED = 42

### (1) 가중치 기반 거리 계산 (weight_distance)

In [ ]:
sae: SparseAutoencoder = load_sae(checkpoint=CHECKPOINT, layer_idx=LAYER, return_ckpt=False)
sae.eval()

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

g = torch.Generator()
g.manual_seed(SEED)

activation_dataset = ActivationDataset(data_dir=f"./data/post_mlp_residual_stream/layer_{LAYER}")
activation_loader = DataLoader(
    activation_dataset,
    batch_size=1,
    generator=g,
    num_workers=0,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sae.to(device)

In [ ]:
_weights = sae.encoder.weight.data.cpu().numpy()

_weights.shape

In [ ]:
weights = []
for feature_index in live_feature_indices:
    weights.append(_weights[feature_index])

weight_matrix = np.array(weights)
weight_distance_matrix = pairwise_distances(weight_matrix, metric="cosine")
print(f"weight_distance_matrix shape: {weight_distance_matrix.shape}")

**weight_distance_matrix 정규화(0~1)**

In [ ]:
min_val = weight_distance_matrix.min()
max_val = weight_distance_matrix.max()
print(f"[DEBUG] 원본 최소값: {min_val:.4f}, 원본 최대값: {max_val:.4f}")

# 정규화
if max_val - min_val > 0:
    normalized_weight_distance = (weight_distance_matrix - min_val) / (max_val - min_val)
else:
    normalized_weight_distance = np.zeros_like(weight_distance_matrix)

print(f"[DEBUG] 정규화된 최소값: {normalized_weight_distance.min():.4f}, 정규화된 최대값: {normalized_weight_distance.max():.4f}")

**예시: 첫 번째 live feature (live_feature_indices[0])의 이웃 찾기**

In [ ]:
# target_local_index = 0 

# # 1. 해당 특징(행)의 거리 값들을 가져옵니다.
# distances_from_target = weight_distance_matrix[target_local_index]

# # 2. 거리(distance)가 낮은 순(가까운 순)으로 "로컬 인덱스"를 정렬합니다.
# # np.argsort는 값 자체 대신, 정렬된 값의 '인덱스'를 반환합니다.
# nearest_neighbor_local_indices = np.argsort(distances_from_target)

# # 3. "로컬 인덱스"를 실제 "feature_index"로 변환합니다.
# k = 5  # K-NN (가장 가까운 5개의 이웃)
# print(f"--- Feature {live_feature_indices[target_local_index]}의 가장 가까운 이웃 특징 ---")

# # nearest_neighbor_local_indices[0]은 항상 자기 자신 (거리 0)입니다.
# # 따라서 [1:] 부터 k+1개까지 슬라이싱하여 실제 이웃을 봅니다.
# for i in range(1, k + 1):
#     local_idx = nearest_neighbor_local_indices[i]
    
#     # "로컬 인덱스"를 "실제 특징 인덱스"로 매핑
#     actual_feature_index = live_feature_indices[local_idx]
#     distance = distances_from_target[local_idx]
    
#     print(f"  {i}순위: Feature {actual_feature_index} (Distance: {distance:.4f})")

### (2) 토큰 활성화 기반 거리 계산(activation_distance)
- 샘플: 1,000 건
- 각 샘플들의 토큰을 concatenate하는 방식

In [ ]:
activation_distance_matrix = get_activation_distance_matrix(
    sae=sae,
    dataloader=activation_loader,
    live_feature_indices=live_feature_indices,
    device=device,
    num_samples=NUM_SAMPLES,
)

In [ ]:
# final_distance_matrix = w1 * weight_distance_matrix + w2 * activation_distance_matrix
final_distance_matrix = w1 * normalized_weight_distance + w2 * activation_distance_matrix

final_distance_matrix.shape

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(range(len(final_distance_matrix[0])), final_distance_matrix[-15] + 0.2, width=1, alpha=0.5, edgecolor='black')
plt.bar(range(len(final_distance_matrix[0])), np.clip(final_distance_matrix[-5] - 0.2, a_min=0, a_max=1.0), width=1, alpha=0.6, edgecolor='black')
plt.show()
plt.close()

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(final_distance_matrix.flatten(), bins=50, edgecolor='black', alpha=0.7)
plt.show()

### DBSCAN 클러스터링

In [ ]:
k = 3  # (min_samples)
sorted_distances = np.sort(final_distance_matrix, axis=1)
k_distances = sorted_distances[:, k - 1]
k_distances_sorted = np.sort(k_distances)

plt.figure(figsize=(10, 6))
plt.plot(k_distances_sorted)
plt.title(f'K-Distance Plot (k = {k - 1})')
plt.xlabel('Features (Sorted by distance)')
plt.ylabel(f'{k - 1}-th Nearest Neighbor Distance')
plt.grid(False)

print("그래프의 '팔꿈치(elbow)' 지점의 Y축 값을 확인하여 'eps'로 사용하세요.")  # 어디보자... 0.5?

In [ ]:
EPS = 0.4
MIN_SAMPLES = 3

dbscan = DBSCAN(
    eps=EPS,
    min_samples=MIN_SAMPLES,
    metric='precomputed',
)

db = dbscan.fit(final_distance_matrix)

labels_ = db.labels_

n_clusters_ = len(set(labels_)) - (1 if -1 in labels_ else 0)
n_noise_ = list(labels_).count(-1)

print(f"총 클러스터 개수 (노이즈 제외): {n_clusters_}")
print(f"노이즈로 분류된 특징 개수: {n_noise_}")

### Feature Clusters (with top-15 Significant Features Protection):

In [ ]:
significance_scores = feature_df["significance"].values

labels = cluster_with_protection(
    distance_matrix=final_distance_matrix,
    significance_scores=significance_scores,
    top_n_to_protect=10, # 상위 n개는 노이즈(-1) 금지
    eps=EPS,
    min_samples=MIN_SAMPLES,
)

In [ ]:
visualize_clusters(final_distance_matrix, labels, live_feature_indices)

In [ ]:
show_heatmap(labels, final_distance_matrix)

In [ ]:
clusters = {}
for local_idx, cluster_id in enumerate(labels):
    actual_feature_index = live_feature_indices[local_idx]
    
    if cluster_id not in clusters:
        clusters[cluster_id] = []
    
    clusters[cluster_id].append(actual_feature_index)

# 결과 출력 (예시)
for cluster_id, features in clusters.items():
    if cluster_id == -1:
        print(f"\n[Noise] (총 {len(features)}개):")
    else:
        print(f"\n[Cluster {cluster_id}] (총 {len(features)}개):")
    print(features)

In [ ]:
clusters = {int(k): v for k,v in clusters.items()}
clusters[1]

In [ ]:
clusters_path = Path(str(settings.feature_cluster_path))
with open(clusters_path, "w") as f:
    json.dump(clusters, f, indent=4)

---

## 주의: 아래 fcluster는 채택하지 않음. 대신 dbscan 채택

In [ ]:
DISTANCE_CUTOFF = 0.8

In [ ]:
feature_labels = [f"#{idx}" for idx in live_feature_indices]

np.fill_diagonal(final_distance_matrix, 0)
condensed_distances = squareform(final_distance_matrix)

Z = linkage(condensed_distances, method='average')

plt.figure(figsize=(20, 12))
plt.title('Feature Clusters Dendrogram', fontsize=20)
plt.xlabel('Features', fontsize=16)
plt.ylabel('Distance', fontsize=16)

dendrogram(
    Z,
    labels=feature_labels,
    leaf_rotation=45,
    leaf_font_size=5,
    color_threshold=DISTANCE_CUTOFF,    # 해당 거리에서 클러스터 색상 구분
)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
cluster_labels = fcluster(Z, t=DISTANCE_CUTOFF, criterion='distance')

print(f"거리 {DISTANCE_CUTOFF} 기준으로 총 {len(set(cluster_labels))}개의 클러스터가 생성되었습니다.")

# --- 클러스터 ID -> Feature Index 사전 생성 ---
feature_clusters = {}

# cluster_labels는 live_feature_indices의 로컬 인덱스와 순서가 동일합니다.
for local_idx, cluster_id in enumerate(cluster_labels):
    # 로컬 인덱스(0, 1, 2...)를 실제 feature_index(예: 120, 512, 1024...)로 변환
    actual_feature_index = live_feature_indices[local_idx]
    
    # cluster_id가 int형인지 확인 (fcluster는 numpy int형을 반환할 수 있음)
    cluster_id_int = int(cluster_id)
    
    if cluster_id_int not in feature_clusters:
        feature_clusters[cluster_id_int] = []
        
    feature_clusters[cluster_id_int].append(actual_feature_index)

---

# Cluster별 word cloud

In [ ]:
feature_to_tokens = collect_cluster_tokens(
    activation_loader=activation_loader,
    sae=sae,
    num_samples=NUM_SAMPLES,
)

In [ ]:
len(clusters[-1])

In [ ]:
show_all_wordclouds(clusters=clusters, feature_to_tokens=feature_to_tokens)